In [ ]:
!pip install transformers datasets pandas torch

In [ ]:
!wget https://www.cs.cmu.edu/~enron/enron_mail_20150507.tar.gz

In [ ]:
!tar -xzf enron_mail_20150507.tar.gz

In [ ]:
import os
print(os.listdir())

In [ ]:
os.listdir("maildir")[:10]

In [ ]:
import os
import pandas as pd

emails = []

for root, dirs, files in os.walk("maildir"):
    for file in files:
        path = os.path.join(root, file)

        try:
            with open(path, "r", errors="ignore") as f:
                text = f.read()

            subject = ""
            body = ""

            if "Subject:" in text:
                subject = text.split("Subject:")[1].split("\n")[0]

            if "\n\n" in text:
                body = text.split("\n\n",1)[1]

            if subject.strip() != "" and body.strip() != "":
                emails.append((body.strip(), subject.strip()))

        except:
            pass

df = pd.DataFrame(emails, columns=["email_body","subject"])

print("Dataset size:", len(df))
df.head()

In [ ]:
df = df.sample(5000, random_state=42)

print(len(df))
df.head()

In [ ]:
df = df.rename(columns={
    "email_body":"input_text",
    "subject":"target_text"
})

df["input_text"] = "generate subject: " + df["input_text"]

df.head()

In [ ]:
from datasets import Dataset

dataset = Dataset.from_pandas(df)

dataset

In [ ]:
from transformers import T5Tokenizer, T5ForConditionalGeneration

model_name = "t5-small"

tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)

print("Model loaded")

In [ ]:
def tokenize(example):

    input_enc = tokenizer(
        example["input_text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

    target_enc = tokenizer(
        example["target_text"],
        padding="max_length",
        truncation=True,
        max_length=32
    )

    input_enc["labels"] = target_enc["input_ids"]

    return input_enc


tokenized_dataset = dataset.map(tokenize)

In [ ]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="results",
    per_device_train_batch_size=4,
    num_train_epochs=1,
    logging_steps=100
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)

trainer.train()

In [ ]:
email = "Please attend the project meeting tomorrow at 10 AM"

input_text = "generate subject: " + email

inputs = tokenizer.encode(input_text, return_tensors="pt")

outputs = model.generate(inputs, max_length=20)

print("Generated Subject:", tokenizer.decode(outputs[0], skip_special_tokens=True))

In [ ]:
email = "The finance report for Q3 is attached please review it before tomorrow"

input_text = "generate subject: " + email

inputs = tokenizer.encode(input_text, return_tensors="pt")

outputs = model.generate(inputs, max_length=20)

print("Generated Subject:", tokenizer.decode(outputs[0], skip_special_tokens=True))

In [ ]:
inputs = tokenizer.encode(input_text, return_tensors="pt")

outputs = model.generate(
    inputs,
    max_length=15,
    num_beams=5,
    no_repeat_ngram_size=2,
    early_stopping=True
)

print("Generated Subject:", tokenizer.decode(outputs[0], skip_special_tokens=True))